In [ ]:
class User:
    def sing(self):
        return "la la la"


class Wizard(User):
    def __init__(self, name):
        self.name = name

    def cast_spell(self):
        return "abracadabra"


wizard = Wizard("Gandalf")
print(wizard.sing())

la la la


In [ ]:
print(isinstance(wizard, Wizard))  # True
print(isinstance(wizard, User))  # True
print(isinstance(wizard, object))  # True
print(isinstance(Wizard, User))  # False

True
True
True
False


In [8]:
print(issubclass(Wizard, User))  # True
print(issubclass(Wizard, object))  # True
print(issubclass(User, User))  # False

True
True
True


The Gist of Inheritance
Inheritance is simply a way to share code. A Child class copies the blueprint of a Parent class so you don't have to rewrite the same methods. If a child needs to do something differently, it overrides the parent's method by writing its own version with the exact same name.

Behind the Scenes: The Memory Architecture
When you write an inheritance hierarchy, Python doesn't actually copy or merge any code in memory. It uses pointers to link namespaces.

Let's use this simple setup:

In [1]:
class Parent:
    def greet(self):
        return "Hello"


class Child(Parent):
    pass


c = Child()

Here is exactly how this looks in RAM:

1. The Class Objects are Created Permanent
When Python executes your script, it allocates memory for two unique Class objects:

Parent Class Object (e.g., at address 0x1111) holds its own namespace dictionary (__dict__), which contains a pointer to the greet function.

Child Class Object (e.g., at address 0x2222) holds an empty namespace dictionary because you didn't write any methods inside it.

2. The Ancestry Link (__bases__)
How does Child know it belongs to Parent? Inside the Child class object at 0x2222, Python automatically populates a hidden attribute called __bases__.

Child.__bases__ holds a tuple containing the memory pointer (0x1111,).

This is a hard, physical link in memory pointing directly up to the Parent class object.

3. The Instance Namespace (c)
When you run c = Child(), Python allocates a tiny block of memory for the instance c (e.g., at address 0x9999).

The instance c does NOT contain any methods. It only contains its own data dictionary (__dict__) for variables like self.name.

Crucially, c contains a hidden pointer called __class__ that points directly to its blueprint: 0x2222 (Child).

The Memory Search Path in Action
When you execute c.greet(), Python follows a strict chain of memory pointers to find that code:

[ Instance 'c' (0x9999) ] 
       | 
       | (Looks for 'greet' in instance dict -> Not found)
       v
[ Class 'Child' (0x2222) ] 
       | 
       | (Looks for 'greet' in Child dict -> Not found)
       v  (__bases__ pointer triggers)
[ Class 'Parent' (0x1111) ] 
       | 
       +--> Found 'greet' function pointer!
Python checks 0x9999 (Instance c). It's not there.

Python uses the __class__ pointer to jump to 0x2222 (Class Child). It's not there.

Python uses the __bases__ pointer to jump to 0x1111 (Class Parent). It finds the function!

The Descriptor Protocol wraps it, binds instance c (0x9999) as self, and executes the C-code.

Inheritance memory management is just delegated lookup. There is no duplicating of functions in RAM; everything uses static pointers up the chain.